In [12]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkTransformationsDemo").getOrCreate()
sc = spark.sparkContext

print("SparkSession Initialized successfully!")

# Create a sample RDD
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(data, 4) # 4 partitions for demonstration
print(f"Original RDD partitions: {rdd.getNumPartitions()}")
print(f"Original RDD content (first 5 elements): {rdd.take(5)}")

print("\n--- Narrow Transformations ---")


SparkSession Initialized successfully!
Original RDD partitions: 4
Original RDD content (first 5 elements): [1, 2, 3, 4, 5]

--- Narrow Transformations ---


In [21]:
# 1. map(func): Applies a function to each element. (Narrow Transformation) / [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mapped_rdd = rdd.map(lambda x: x * 2)
print(f"Map Transformation (multiply by 2): {mapped_rdd}")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonUtils.getBroadcastThreshold.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.api.java.JavaSparkContext.sc()" because "jsc" is null
	at org.apache.spark.api.java.JavaSparkContext$.toSparkContext(JavaSparkContext.scala:822)
	at org.apache.spark.api.python.PythonUtils$.getBroadcastThreshold(PythonUtils.scala:93)
	at org.apache.spark.api.python.PythonUtils.getBroadcastThreshold(PythonUtils.scala)
	at jdk.internal.reflect.GeneratedMethodAccessor48.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py

In [14]:
# 2. filter(func): Selects elements for which the function returns true. (Narrow Transformation)/ [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
filtered_rdd = rdd.filter(lambda x: x > 5)
print(f"Filter Transformation (elements > 5): {filtered_rdd.collect()}")

Filter Transformation (elements > 5): [6, 7, 8, 9, 10]


In [15]:
# 3. flatMap(func): Maps each element to zero or more output elements. (Narrow Transformation)
string_rdd = sc.parallelize(["hello world", "spark demo", "transformations"])
flat_mapped_rdd = string_rdd.flatMap(lambda x: x.split(" "))
print(f"FlatMap Transformation (split words): {flat_mapped_rdd.collect()}")

FlatMap Transformation (split words): ['hello', 'world', 'spark', 'demo', 'transformations']


In [16]:
# 4. mapPartitions(func): Applies a function to each partition. (Narrow Transformation)/ [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# Useful for initializing per-partition setup (e.g., database connections)
def process_partition(iterator):
    # Simulate some setup per partition
    # print("Processing a partition...") # Uncomment to see it in action
    return [x * 10 for x in iterator]

map_partitions_rdd = rdd.mapPartitions(process_partition)
print(f"MapPartitions Transformation (multiply by 10 per partition): {map_partitions_rdd.collect()}")

MapPartitions Transformation (multiply by 10 per partition): [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


In [17]:
# 5. mapPartitionsWithIndex(func): Applies a function to each partition, including partition index. (Narrow Transformation)/ [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
def process_partition_with_index(index, iterator):
    return [f"Partition {index}: {x}" for x in iterator]

map_partitions_with_index_rdd = rdd.mapPartitionsWithIndex(process_partition_with_index)
print(f"MapPartitionsWithIndex Transformation: {map_partitions_with_index_rdd.collect()}")

print("\n--- Wide Transformations ---")

MapPartitionsWithIndex Transformation: ['Partition 0: 1', 'Partition 0: 2', 'Partition 1: 3', 'Partition 1: 4', 'Partition 2: 5', 'Partition 2: 6', 'Partition 3: 7', 'Partition 3: 8', 'Partition 3: 9', 'Partition 3: 10']

--- Wide Transformations ---


In [18]:
# 6. coalesce(numPartitions): Reduces the number of partitions. (Can be Wide if shuffling is required)/ [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# If decreasing partitions, it tries to avoid a full shuffle if possible.
# Here we reduce from 4 to 2 partitions, which might not require a full shuffle depending on data distribution.
coalesced_rdd_less = rdd.coalesce(2)
print(f"Coalesce Transformation (reduce to 2 partitions): {coalesced_rdd_less.getNumPartitions()}")
# To force a shuffle even when decreasing, set `shuffle=True`
coalesced_rdd_shuffle = rdd.coalesce(2, shuffle=True)
print(f"Coalesce Transformation (reduce to 2 partitions with shuffle): {coalesced_rdd_shuffle.getNumPartitions()}")

# If increasing partitions, coalesce implicitly performs a shuffle to redistribute data.
# Note: repartition is generally preferred for increasing partitions explicitly as it is clearer.
coalesced_rdd_more = rdd.coalesce(6, shuffle=True) # Explicitly shuffling to increase partitions effectively
print(f"Coalesce Transformation (increase to 6 partitions with shuffle): {coalesced_rdd_more.getNumPartitions()}")

Coalesce Transformation (reduce to 2 partitions): 2
Coalesce Transformation (reduce to 2 partitions with shuffle): 2
Coalesce Transformation (increase to 6 partitions with shuffle): 6


In [19]:
# 7. repartition(numPartitions): Increases or decreases the number of partitions. (Always Wide Transformation - involves a shuffle) / [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
repartitioned_rdd_more = rdd.repartition(6)
print(f"Repartition Transformation (increase to 6 partitions): {repartitioned_rdd_more.getNumPartitions()}")

repartitioned_rdd_less = rdd.repartition(2)
print(f"Repartition Transformation (decrease to 2 partitions): {repartitioned_rdd_less.getNumPartitions()}")

Repartition Transformation (increase to 6 partitions): 6
Repartition Transformation (decrease to 2 partitions): 2


In [20]:
# Stop SparkSession
spark.stop()
print("\nSparkSession stopped.")


SparkSession stopped.
